# Import Modules 

In [15]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Load Data 

In [16]:
# 1. Load Data
train_df = pd.read_csv('dataset/public/train.csv')
test_df = pd.read_csv('dataset/public/test.csv')
# Rename label column
train_df = train_df.rename(columns={"target": "labels"})

# Tokenization 

In [17]:
# 2. Tokenization
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Prepare dataset (simplified for notebook)
# Convert pandas to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/162 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

In [18]:
tokenized_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

# Model Setup

In [19]:
# 3. Model Setup
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Training Arguments & Training

In [20]:
# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=50,
    weight_decay=0.01,
    report_to='none'
)

# 5. Fine-tune
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
)

trainer.train()

C:\Users\ttarv\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=210, training_loss=0.047282327924455914, metrics={'train_runtime': 4759.8234, 'train_samples_per_second': 0.34, 'train_steps_per_second': 0.044, 'total_flos': 214597185822720.0, 'train_loss': 0.047282327924455914, 'epoch': 10.0})

# Predict & Test 

In [22]:
# 6. Predict on Test
import numpy as np
predictions = trainer.predict(tokenized_test)
# The output is logits; we take the index of the max logit
test_df['target'] = np.argmax(predictions.predictions, axis=1)

# Format Output

In [23]:
# 7. Save Submission
test_df[['id', 'target']].to_csv('working/submission.csv', index=False)
print("Submission file saved successfully.")

Submission file saved successfully.
